In [0]:
from pyspark.sql import functions as F

In [0]:
def enriched_orders(transformed_orders_path,enriched_orders_path,enriched_customers_path,enriched_products_path):
    try:

        df_orders= spark.read.format("delta").load(transformed_orders_path)
        order_date_format = "d/M/yyyy"
        df_orders = df_orders.withColumn("Order_Date", F.to_date(F.col("Order_Date"), order_date_format))\
                                   .withColumn("Profit", F.round(F.coalesce(F.col("Profit"), F.lit(0.0)), 2))

        df_customers = spark.read.format("delta").load(enriched_customers_path)
        df_products = spark.read.format("delta").load(enriched_products_path)
        df_enriched = df_orders.join(df_customers, on="Customer_ID", how="left")\
                               .join(df_products, on="Product_ID", how="left")
        df_enriched = df_enriched.withColumnRenamed("Category","Product_Category")\
                                 .withColumnRenamed("Sub-Category","Product_Sub_Category")
        col_list_select = ['Order_ID','Row_ID','Order_Date','Profit','Customer_Name','Country','Product_Category','Product_Sub_Category']
        df_enriched = df_enriched.select(col_list_select).dropDuplicates()
        df_enriched = df_enriched.fillna('Unknown')
        df_enriched.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(enriched_orders_path)
    except Exception as e:
        print(f"someting went wrong {e}")

In [0]:
transformed_orders_path = "/Volumes/workspace/default/transformed/Orders"
enriched_orders_path = "/Volumes/workspace/default/enriched/Orders"
enriched_customers_path = "/Volumes/workspace/default/enriched/Customers"
enriched_products_path = "/Volumes/workspace/default/enriched/Products"

enriched_orders(transformed_orders_path,enriched_orders_path,enriched_customers_path,enriched_products_path)